In [2]:
import numpy as np
import pandas as pd
import plotly.express as px

## [1] Data Loading

In [3]:
df = pd.read_csv('data/restaurant_sales_data.csv')

## [2] Data Understanding

In [13]:
df.shape

(17534, 9)

In [4]:
df.head(5)

,Order ID,Customer ID,Category,Item,Price,Quantity,Order Total,Order Date,Payment Method
0,ORD_705844,CUST_092,Side Dishes,Side Salad,3.0,1.0,3.0,2023-12-21,Credit Card
1,ORD_338528,CUST_021,Side Dishes,Mashed Potatoes,4.0,3.0,12.0,2023-05-19,Digital Wallet
2,ORD_443849,CUST_029,Main Dishes,Grilled Chicken,15.0,4.0,60.0,2023-09-27,Credit Card
3,ORD_630508,CUST_075,Drinks,NaN,NaN,2.0,5.0,2022-08-09,Credit Card
4,ORD_648269,CUST_031,Main Dishes,Pasta Alfredo,12.0,4.0,48.0,2022-05-15,Cash


In [ ]:
df.info()
# There are missing values at these columns [Item, Price, Quantity, Order Total, Payment Method]
# Columns with their datatypes have to be modified [Order Date]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17534 entries, 0 to 17533
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Order ID        17534 non-null  object 
 1   Customer ID     17534 non-null  object 
 2   Category        17534 non-null  object 
 3   Item            15776 non-null  object 
 4   Price           16658 non-null  float64
 5   Quantity        17104 non-null  float64
 6   Order Total     17104 non-null  float64
 7   Order Date      17534 non-null  object 
 8   Payment Method  16452 non-null  object 
dtypes: float64(3), object(6)
memory usage: 1.2+ MB


In [9]:
df.describe()
# [Price, Order Total] have outliers

,Price,Quantity,Order Total
count,16658.000000,17104.000000,17104.000000
mean,6.586325,3.014149,19.914494
std,4.834652,1.414598,18.732549
min,1.000000,1.000000,1.000000
25%,3.000000,2.000000,7.500000
50%,5.000000,3.000000,15.000000
75%,7.000000,4.000000,25.000000
max,20.000000,5.000000,100.000000


## [3] Data Cleaning

### 1. Standardize the columns names

In [11]:
df.columns = df.columns.str.lower().str.strip().str.replace(' ', '_')

In [12]:
df.head(5)

,order_id,customer_id,category,item,price,quantity,order_total,order_date,payment_method
0,ORD_705844,CUST_092,Side Dishes,Side Salad,3.0,1.0,3.0,2023-12-21,Credit Card
1,ORD_338528,CUST_021,Side Dishes,Mashed Potatoes,4.0,3.0,12.0,2023-05-19,Digital Wallet
2,ORD_443849,CUST_029,Main Dishes,Grilled Chicken,15.0,4.0,60.0,2023-09-27,Credit Card
3,ORD_630508,CUST_075,Drinks,NaN,NaN,2.0,5.0,2022-08-09,Credit Card
4,ORD_648269,CUST_031,Main Dishes,Pasta Alfredo,12.0,4.0,48.0,2022-05-15,Cash


### 2. Drop Unuseful columns for Analysis

In [16]:
df.drop(columns=['order_id'], inplace=True)

In [17]:
df.head(5)

,customer_id,category,item,price,quantity,order_total,order_date,payment_method
0,CUST_092,Side Dishes,Side Salad,3.0,1.0,3.0,2023-12-21,Credit Card
1,CUST_021,Side Dishes,Mashed Potatoes,4.0,3.0,12.0,2023-05-19,Digital Wallet
2,CUST_029,Main Dishes,Grilled Chicken,15.0,4.0,60.0,2023-09-27,Credit Card
3,CUST_075,Drinks,NaN,NaN,2.0,5.0,2022-08-09,Credit Card
4,CUST_031,Main Dishes,Pasta Alfredo,12.0,4.0,48.0,2022-05-15,Cash


### 3. Handle missing values

In [18]:
numerical = ['price', 'quantity', 'order_total']
categorical = ['customer_id', 'category', 'item', 'order_date']

In [20]:
df.isna().sum().sort_values(ascending=False)

item              1758
payment_method    1082
price              876
quantity           430
order_total        430
category             0
customer_id          0
order_date           0
dtype: int64

In [22]:
for col in numerical:
    fig = px.box(df, x=col, title=f'{col} boxblot')
    fig.show()

In [29]:
def calc_outliers_perct(col):
    col = col.dropna()

    q1 = col.quantile(0.25)
    q3 = col.quantile(0.75)
    iqr = q3-q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    return ((col < lower_bound) | (col > upper_bound)).mean() * 100

In [30]:
outliers_in_perct = {col : calc_outliers_perct(df[col]) for col in numerical}

In [31]:
outliers_in_perct

{'price': np.float64(13.873214071317086),
 'quantity': np.float64(0.0),
 'order_total': np.float64(8.237839101964454)}

In [32]:
for col in numerical:
    df[col] = df[col].fillna(df[col].median())

In [34]:
df.isna().sum().sort_values(ascending=False)

item              1758
payment_method    1082
category             0
customer_id          0
price                0
quantity             0
order_total          0
order_date           0
dtype: int64

In [35]:
payment_mode = df['payment_method'].mode()[0]
df['payment_method'] = df['payment_method'].fillna(payment_mode)

In [39]:
df['item'] = df.groupby('category')['item'].transform(
    lambda category: category.fillna(category.mode()[0] if not category.mode().empty else "Unknown")
)

In [40]:
df.isna().sum().sort_values(ascending=False)

customer_id       0
category          0
item              0
price             0
quantity          0
order_total       0
order_date        0
payment_method    0
dtype: int64

### 4. Fix DataTypes

In [45]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17534 entries, 0 to 17533
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   customer_id     17534 non-null  object 
 1   category        17534 non-null  object 
 2   item            17534 non-null  object 
 3   price           17534 non-null  float64
 4   quantity        17534 non-null  float64
 5   order_total     17534 non-null  float64
 6   order_date      17534 non-null  object 
 7   payment_method  17534 non-null  object 
dtypes: float64(3), object(5)
memory usage: 1.1+ MB


In [47]:
df.order_date.unique()

array(['2023-12-21', '2023-05-19', '2023-09-27', '2022-08-09',
       '2022-05-15', '2022-07-20', '2022-08-19', '2023-02-15',
       '2023-12-16', '2022-08-07', '2023-12-09', '2023-10-30',
       '2023-04-11', '2023-08-18', '2022-06-08', '2022-08-28',
       '2022-05-01', '2023-09-01', '2022-11-13', '2023-06-08',
       '2023-02-21', '2023-02-01', '2023-05-09', '2023-09-14',
       '2023-03-18', '2023-05-18', '2023-09-05', '2022-02-02',
       '2023-07-30', '2023-01-31', '2023-02-28', '2022-11-19',
       '2023-05-03', '2023-04-20', '2023-09-18', '2022-04-08',
       '2023-07-29', '2023-10-16', '2022-02-09', '2023-08-17',
       '2022-04-13', '2023-07-21', '2023-12-03', '2023-09-16',
       '2023-02-11', '2023-02-05', '2022-01-16', '2023-11-21',
       '2022-10-08', '2023-03-01', '2023-05-23', '2022-08-11',
       '2022-08-03', '2022-12-20', '2022-09-12', '2022-06-24',
       '2023-08-04', '2022-03-11', '2023-09-28', '2022-08-12',
       '2023-01-23', '2023-07-08', '2022-12-07', '2023-

In [48]:
df['order_date'] = pd.to_datetime(df['order_date'])

In [54]:
df['order_year'] = df['order_date'].dt.year
df['order_month'] = df['order_date'].dt.month
df['order_day'] = df['order_date'].dt.day
df['order_day_of_week'] = df['order_date'].dt.dayofweek
df['order_day_name'] = df['order_date'].dt.day_name()
df['is_weekend'] = df['order_day_of_week'].isin([5, 6]).astype(int)

In [55]:
df.head(10)

,customer_id,category,item,price,quantity,order_total,order_date,payment_method,order_year,order_month,order_day,order_day_of_week,order_day_name,is_weekend
0,CUST_092,Side Dishes,Side Salad,3.0,1.0,3.0,2023-12-21,Credit Card,2023,12,21,3,Thursday,0
1,CUST_021,Side Dishes,Mashed Potatoes,4.0,3.0,12.0,2023-05-19,Digital Wallet,2023,5,19,4,Friday,0
2,CUST_029,Main Dishes,Grilled Chicken,15.0,4.0,60.0,2023-09-27,Credit Card,2023,9,27,2,Wednesday,0
3,CUST_075,Drinks,Water,5.0,2.0,5.0,2022-08-09,Credit Card,2022,8,9,1,Tuesday,0
4,CUST_031,Main Dishes,Pasta Alfredo,12.0,4.0,48.0,2022-05-15,Cash,2022,5,15,6,Sunday,1
5,CUST_031,Main Dishes,Salmon,18.0,5.0,90.0,2022-07-20,Digital Wallet,2022,7,20,2,Wednesday,0
6,CUST_071,Side Dishes,Garlic Bread,4.0,5.0,20.0,2022-08-19,Credit Card,2022,8,19,4,Friday,0
7,CUST_077,Main Dishes,Pasta Alfredo,15.0,3.0,45.0,2023-02-15,Cash,2023,2,15,2,Wednesday,0
8,CUST_083,Desserts,Ice Cream,6.0,2.0,12.0,2023-12-16,Cash,2023,12,16,5,Saturday,1
9,CUST_085,Main Dishes,Vegetarian Platter,14.0,5.0,70.0,2022-08-07,Credit Card,2022,8,7,6,Sunday,1


### 5. Remove Duplicates - if existed

In [ ]:
df.duplicated().sum()

np.int64(6)

In [59]:
df.drop_duplicates(inplace=True)

In [60]:
df.duplicated().sum()

np.int64(0)

## [4] EDA